In [ ]:
import bpy
import random
from pyglm import glm

image = bpy.Image("/Users/ajayvenkat/Development/metal-procedural-art/test.png", glm.vec2(100,100))
col = (random.random(), random.random(), random.random())
square = bpy.Square(glm.vec2(0,0), glm.vec4(col[0], col[1], col[2], 1.0))
circle = bpy.Circle(glm.vec2(0,0), glm.vec4(col[0], col[1], col[2], 1.0))

In [6]:
import time
import random
import bpy
import math

canvas = bpy.Canvas(800, 600)
print("Rendering artwork...")
layer = bpy.Layer()

canvas.add_layer(layer)

perc = 0

while True:
    cur = glm.vec3(math.sin(perc)*random.random()*500, math.cos(perc)*random.random()*500, 0)

    for i in range(200):
        # random color
        col = (random.random(), random.random(), random.random())
        size = glm.vec2(250*math.cos(perc*10), 250*math.cos(perc*10))

        square.color = glm.vec4(col[0], col[1], col[2], 1.0)
        square.size = size
        circle.color = glm.vec4(col[0], col[1], col[2], 1.0)    
        circle.size = size
        image.set_size(size)

        if i % 100 == 0:
            target = glm.vec3(random.random()*800, random.random()*600, 0)

        cur = glm.lerp(cur, target, 0.01)
        mat = glm.mat4(1.0)
        perc = float(i)/200.0
        mat = glm.translate(mat, cur)
        mat = glm.rotate(mat, perc*math.pi*2, glm.vec3(0, 0, 1))
        mat2 = glm.inverse(mat)
        if i % 2 != 0:
            layer.draw(square, mat)
        else:
            layer.draw(circle, mat)
        # time.sleep(0.01)
        canvas.render_out("/Users/ajayvenkat/Development/metal-procedural-art/artworks/test.png")

    
    



Rendering artwork...


AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_bytes_per_row
AGX: Texture read/write assertion failed: bytes_per_row >= used_

KeyboardInterrupt: 

In [ ]:
import bpy


In [23]:
import bpy
from pyglm import glm
image = bpy.Image("/Users/ajayvenkat/Development/metal-procedural-art/test.png", glm.vec2(300,300))
square = bpy.Square(glm.vec2(0,0), glm.vec4(col[0], col[1], col[2], 1.0))
circle = bpy.Circle(glm.vec2(0,0), glm.vec4(col[0], col[1], col[2], 1.0))
canvas = bpy.Canvas(800, 600)
print("Rendering artwork...")
layer = bpy.Layer()
canvas.add_layer(layer)

for f in range(120):
    mat = glm.mat4(1.0)
    mat = glm.translate(mat, glm.vec3(800, 300, 0))

    layer.draw(bpy.tint(image, glm.vec4(1.0, 0.0, 0.0, 0.5)), mat)

    # slowly translate to the left and shift the tint color from red to blue over time
    for i in range(500):
        perc = float(i)/200.0
        mat = glm.mat4(1.0)
        mat = glm.translate(mat, glm.vec3(800 - perc*800, 300, 0))
        mat = glm.rotate(mat, perc*math.pi*.50*f, glm.vec3(0, 0, 1))
        mat = glm.scale(mat, glm.vec3(1.0 + perc, 1.0 + perc, 1.0))
        tintColor = glm.vec4(1.0 - perc, 0.0, perc, 0.5)
        layer.draw(bpy.tint(image, tintColor), mat)

    canvas.render_out(f"/Users/ajayvenkat/Development/metal-procedural-art/artworks/test_vid/test_{f}.png")

Rendering artwork...


In [1]:
import bpy
from pyglm import glm
import math
import random
import colorsys

W, H = 800, 600
CX, CY = W / 2, H / 2
DIAG = math.hypot(CX, CY)  # distance center -> corner, for full-bleed coverage

random.seed(42)

canvas = bpy.Canvas(W, H)
print("Rendering artwork...")
layer = bpy.Layer()
canvas.add_layer(layer)

image = bpy.Image("/Users/ajayvenkat/Development/metal-procedural-art/test.png", glm.vec2(300, 300))


def hsv(h, s, v, a=1.0):
    r, g, b = colorsys.hsv_to_rgb(h % 1.0, s, v)
    return glm.vec4(r, g, b, a)


def mat_at(pos, rot=0.0, scale=1.0):
    m = glm.mat4(1.0)
    m = glm.translate(m, glm.vec3(pos.x, pos.y, 0))
    m = glm.rotate(m, rot, glm.vec3(0, 0, 1))
    if isinstance(scale, (tuple, list)) or hasattr(scale, "x"):
        m = glm.scale(m, glm.vec3(scale.x, scale.y, 1))
    else:
        m = glm.scale(m, glm.vec3(scale, scale, 1))
    return m


# ---------------------------------------------------------------
# LAYER 1 — full-bleed background field: warped grid of squares,
# hue driven by position + a sine warp so it ripples like a heat haze
# ---------------------------------------------------------------
GRID_X, GRID_Y = 22, 16
cell_w, cell_h = W / GRID_X, H / GRID_Y
for gy in range(GRID_Y):
    for gx in range(GRID_X):
        px = gx * cell_w + cell_w / 2
        py = gy * cell_h + cell_h / 2

        warp = 10 * math.sin(gx * 0.6 + gy * 0.4)
        pos = glm.vec2(px + warp, py + warp * 0.6)

        hue = (gx / GRID_X + gy / GRID_Y * 0.5) % 1.0
        col = hsv(hue, 0.65, 0.55, 0.35)

        rot = (gx + gy) * 0.15
        sq_scale = cell_w * 0.42

        square = bpy.Square(glm.vec2(0, 0), col)
        layer.draw(square, mat_at(pos, rot, sq_scale))


# ---------------------------------------------------------------
# LAYER 2 — scattered particle field across the whole canvas,
# varied sizes, low-mid alpha, covers corners the center pieces miss
# ---------------------------------------------------------------
N_SCATTER = 90
for i in range(N_SCATTER):
    pos = glm.vec2(random.uniform(0, W), random.uniform(0, H))
    hue = random.random()
    col = hsv(hue, 0.85, 1.0, random.uniform(0.15, 0.45))
    scale = random.uniform(4, 22)
    circle = bpy.Circle(glm.vec2(0, 0), col)
    layer.draw(circle, mat_at(pos, 0.0, scale))


# ---------------------------------------------------------------
# LAYER 3 — diagonal flowing wave bands, corner to corner,
# squares riding two crossing sine paths for a woven look
# ---------------------------------------------------------------
def wave_band(n, amp, freq, phase, hue_base, y_offset, thickness):
    for i in range(n):
        t = i / n
        x = t * W
        y = y_offset + amp * math.sin(t * freq * 2 * math.pi + phase)
        hue = (hue_base + t * 0.4) % 1.0
        col = hsv(hue, 0.9, 1.0, 0.5)
        rot = math.cos(t * freq * 2 * math.pi + phase) * 0.8
        square = bpy.Square(glm.vec2(0, 0), col)
        layer.draw(square, mat_at(glm.vec2(x, y), rot, thickness))


wave_band(60, 90, 3.0, 0.0, 0.55, H * 0.3, 10)
wave_band(60, 70, 4.0, math.pi / 2, 0.05, H * 0.72, 8)


# ---------------------------------------------------------------
# LAYER 4 — four corner spiral bursts (variations on the original
# ring), pushed toward each corner so color reaches every edge
# ---------------------------------------------------------------
def spiral_burst(center, n, base_radius, hue_base, dir_sign=1):
    for i in range(n):
        t = i / n
        angle = dir_sign * t * 3 * math.pi
        radius = base_radius * (0.3 + 0.7 * t)
        pos = center + glm.vec2(math.cos(angle), math.sin(angle)) * radius
        hue = (hue_base + t) % 1.0
        col = hsv(hue, 0.9, 1.0, 0.5)
        scale = 8 + 14 * (1 - t)
        circle = bpy.Circle(glm.vec2(0, 0), col)
        layer.draw(circle, mat_at(pos, angle, scale))


spiral_burst(glm.vec2(0, 0), 40, 220, 0.0, 1)
spiral_burst(glm.vec2(W, 0), 40, 220, 0.25, -1)
spiral_burst(glm.vec2(0, H), 40, 220, 0.5, -1)
spiral_burst(glm.vec2(W, H), 40, 220, 0.75, 1)


# ---------------------------------------------------------------
# LAYER 5 — central mandala: rainbow ring, counter-rotating
# squares, kaleidoscope image burst, soft halo glow
# ---------------------------------------------------------------
N_RING = 32
for i in range(N_RING):
    t = i / N_RING
    angle = t * 2 * math.pi
    hue = t
    col = hsv(hue, 0.9, 1.0, 0.6)
    radius = 260 + 50 * math.sin(t * 8 * math.pi)
    pos = glm.vec2(CX, CY) + glm.vec2(math.cos(angle), math.sin(angle)) * radius
    scale = 26 + 18 * math.sin(t * 12 * math.pi)
    circle = bpy.Circle(glm.vec2(0, 0), col)
    layer.draw(circle, mat_at(pos, angle * 3, scale))

N_SQ = 20
for i in range(N_SQ):
    t = i / N_SQ
    angle = -t * 2 * math.pi
    hue = (t + 0.5) % 1.0
    col = hsv(hue, 1.0, 0.9, 0.7)
    radius = 120 + 30 * math.cos(t * 6 * math.pi)
    pos = glm.vec2(CX, CY) + glm.vec2(math.cos(angle), math.sin(angle)) * radius
    square = bpy.Square(glm.vec2(0, 0), col)
    layer.draw(square, mat_at(pos, angle * -5, 20))

N_IMG = 10
for i in range(N_IMG):
    t = i / N_IMG
    angle = t * 2 * math.pi
    hue = t
    tint_col = hsv(hue, 0.8, 1.0, 0.4)
    m = glm.mat4(1.0)
    m = glm.translate(m, glm.vec3(CX, CY, 0))
    m = glm.rotate(m, angle, glm.vec3(0, 0, 1))
    m = glm.translate(m, glm.vec3(70, 0, 0))
    m = glm.rotate(m, angle * 2, glm.vec3(0, 0, 1))
    m = glm.scale(m, glm.vec3(0.45, 0.45, 1))
    layer.draw(bpy.tint(image, tint_col), m)

N_HALO = 7
for i in range(N_HALO):
    t = i / N_HALO
    hue = (t * 0.3 + 0.6) % 1.0
    col = hsv(hue, 0.7, 1.0, 0.10)
    circle = bpy.Circle(glm.vec2(0, 0), col)
    layer.draw(circle, mat_at(glm.vec2(CX, CY), 0.0, 300 + i * 30))


# ---------------------------------------------------------------
# LAYER 6 — four small satellite kaleidoscopes orbiting the
# midpoints of each edge, tying the corners back to the center
# ---------------------------------------------------------------
edge_points = [
    glm.vec2(CX, 60),      # top
    glm.vec2(CX, H - 60),  # bottom
    glm.vec2(60, CY),      # left
    glm.vec2(W - 60, CY),  # right
]
for j, ep in enumerate(edge_points):
    hue_base = j / len(edge_points)
    for i in range(10):
        t = i / 10
        angle = t * 2 * math.pi
        hue = (hue_base + t * 0.3) % 1.0
        col = hsv(hue, 0.9, 1.0, 0.55)
        radius = 40 + 15 * math.sin(t * 6 * math.pi)
        pos = ep + glm.vec2(math.cos(angle), math.sin(angle)) * radius
        circle = bpy.Circle(glm.vec2(0, 0), col)
        layer.draw(circle, mat_at(pos, angle * 2, 8))


canvas.render_out("/Users/ajayvenkat/Development/metal-procedural-art/artworks/trippy2.png")
print("Done.")

BrushPY Engine ready, created Command Queue & Shader Library
Rendering artwork...
Done.
